In [15]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
import warnings
warnings.filterwarnings('ignore')

print("--- 🏁 PHASE 4: HYBRID BLENDER ENVIRONMENT FULLY LOADED ---")

--- 🏁 PHASE 4: HYBRID BLENDER ENVIRONMENT FULLY LOADED ---


In [16]:
print("Loading catalog datasets...")

# 1. Load the filtered datasets from your local processed directory
movies = pd.read_csv('/Users/dhruvitjalodhara/programming/ML Practice/Movie Recommendation System/movie dataset/processed/filtered_movies.csv')
ratings = pd.read_csv('/Users/dhruvitjalodhara/programming/ML Practice/Movie Recommendation System/movie dataset/processed/filtered_ratings.csv')

print(f"✅ Master Movies Shape  : {movies.shape}")
print(f"✅ User Ratings Shape   : {ratings.shape}")

Loading catalog datasets...
✅ Master Movies Shape  : (16116, 5)
✅ User Ratings Shape   : (29657653, 3)


In [17]:
print("Building Collaborative Filtering Engine...")

# 1. Pivot the flat ratings dataframe to map items vs users
pivoted_ratings = ratings.pivot(index='movieId', columns='userId', values='rating')
pivoted_ratings = pivoted_ratings.fillna(0)

# 2. Compress the pivot framework into a memory-efficient Compressed Sparse Row matrix
ratings_csr = csr_matrix(pivoted_ratings.values)

# 3. Fit our Tournament Champion Model (TruncatedSVD)
svd = TruncatedSVD(n_components=50, random_state=42)
movie_features_svd = svd.fit_transform(ratings_csr)

# 4. Generate the complete Collaborative Pairwise Similarity Matrix
svd_sim_matrix = cosine_similarity(movie_features_svd)

print(f"✅ Collaborative SVD Feature Matrix Shape   : {movie_features_svd.shape}")
print(f"✅ Collaborative Pairwise Similarity Shape  : {svd_sim_matrix.shape}")

Building Collaborative Filtering Engine...
✅ Collaborative SVD Feature Matrix Shape   : (16116, 50)
✅ Collaborative Pairwise Similarity Shape  : (16116, 16116)


In [18]:
print("Building Content-Based NLP Filtering Engine...")

# 1. Perform defensive data cleaning on text descriptors
movies['metadata_soup'] = movies['metadata_soup'].fillna('')

# 2. Instantiate and fit the unique-word frequency mapping model
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['metadata_soup'])

# 3. Compute text-based similarity matrix using ultra-fast linear_kernel dot product 
content_sim_matrix = linear_kernel(tfidf_matrix, tfidf_matrix)

print(f"✅ Content TF-IDF Vocabulary Space Size    : {tfidf_matrix.shape[1]} unique features")
print(f"✅ Content Pairwise Similarity Shape       : {content_sim_matrix.shape}")

Building Content-Based NLP Filtering Engine...
✅ Content TF-IDF Vocabulary Space Size    : 124302 unique features
✅ Content Pairwise Similarity Shape       : (16116, 16116)


In [19]:
# Build reference maps to bridge human-readable text titles with internal matrix offsets
movie_to_idx = pd.Series(movies.index, index=movies['title'])
idx_to_movie = {v: k for k, v in movie_to_idx.items()}

print(f"✅ Successfully mapped {len(movie_to_idx)} titles to positional vector coordinates.")

✅ Successfully mapped 16116 titles to positional vector coordinates.


In [20]:
def get_hybrid_recommendations(movie_title, alpha=0.5, top_n=5):
    """
    Combines Collaborative (SVD) and Content-Based (TF-IDF) scores linearly.
    
    Parameters:
    -----------
    movie_title : str -> Exact string title of the movie query
    alpha       : float -> Weight modifier between 0.0 and 1.0
                           Higher Alpha (e.g. 0.8) -> Favors collaborative user tracking patterns
                           Lower Alpha (e.g. 0.2)  -> Favors contextual plot summaries & genres
    top_n       : int -> Quantified number of final recommendations to yield
    """
    # 1. Validation check to see if the search string is within data parameters
    if movie_title not in movie_to_idx.index:
        print(f"❌ '{movie_title}' not found in the dataset matrix indexes.")
        return None
        
    # 2. Extract the target movie matrix row location
    movie_idx = movie_to_idx[movie_title]
    
    # 3. Isolate the exact similarity score arrays from both engine architectures
    coll_scores = svd_sim_matrix[movie_idx]
    content_scores = content_sim_matrix[movie_idx]
    
    # 4. Apply the Linear Combination blending logic formula
    hybrid_scores = (alpha * coll_scores) + ((1 - alpha) * content_scores)
    
    # 5. Extract positional ranking array sorted from highest match to lowest
    sorted_indices = np.argsort(hybrid_scores)[::-1]
    
    print(f"\n🌪️  HYBRID RECOMMENDER RESULTS FOR: '{movie_title}'")
    print(f"🎛️  CONFIGURATION WEIGHT BALANCER  : Alpha = {alpha}")
    print("=" * 75)
    
    rank = 1
    for idx in sorted_indices:
        # Prevent the engine from recommending the user's input item back to them
        if idx == movie_idx:
            continue
            
        recommended_title = idx_to_movie[idx]
        final_score = hybrid_scores[idx]
        
        # Pull distinct engine weights for full processing transparency
        c_score = coll_scores[idx]
        t_score = content_scores[idx]
        
        print(f"{rank}. {recommended_title}")
        print(f"   [Combined: {final_score:.2%} | SVD (Behavior): {c_score:.2%} | TF-IDF (Text): {t_score:.2%}]")
        print("-" * 75)
        
        rank += 1
        if rank > top_n:
            break

In [21]:
# Target operational query title definition
test_movie = "Kung Fu Panda (2008)"

# Scenario A: Standard 50/50 Balanced Recommendation Profile
get_hybrid_recommendations(test_movie, alpha=0.5, top_n=3)

print("\n" + "#" * 75 + "\n")

# Scenario B: Behavior-Heavy Configuration (80% SVD, 20% Text)
# Captures shared hidden community tastes and user rating alignment habits
get_hybrid_recommendations(test_movie, alpha=0.8, top_n=3)

print("\n" + "#" * 75 + "\n")

# Scenario C: Context-Heavy Configuration (20% SVD, 80% Text)
# Isolates direct narrative plots, storyline extensions, and content sequels
get_hybrid_recommendations(test_movie, alpha=0.2, top_n=3)


🌪️  HYBRID RECOMMENDER RESULTS FOR: 'Kung Fu Panda (2008)'
🎛️  CONFIGURATION WEIGHT BALANCER  : Alpha = 0.5
1. Kung Fu Panda 3 (2016)
   [Combined: 67.46% | SVD (Behavior): 82.62% | TF-IDF (Text): 52.30%]
---------------------------------------------------------------------------
2. Kung Fu Panda 2 (2011)
   [Combined: 67.41% | SVD (Behavior): 90.39% | TF-IDF (Text): 44.43%]
---------------------------------------------------------------------------
3. Ratatouille (2007)
   [Combined: 64.29% | SVD (Behavior): 93.61% | TF-IDF (Text): 34.96%]
---------------------------------------------------------------------------

###########################################################################


🌪️  HYBRID RECOMMENDER RESULTS FOR: 'Kung Fu Panda (2008)'
🎛️  CONFIGURATION WEIGHT BALANCER  : Alpha = 0.8
1. Despicable Me (2010)
   [Combined: 82.55% | SVD (Behavior): 95.99% | TF-IDF (Text): 28.76%]
---------------------------------------------------------------------------
2. Megamind (2010)